[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml/cours/seance4_cours.ipynb)

# Séance 4.4 — Segmenter sans étiquette — quatre clients, quatre traitements

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- distinguer un problème supervisé d'un problème non supervisé
- expliquer pourquoi il faut standardiser avant de mesurer une distance
- appliquer `KMeans` et choisir le nombre de groupes
- donner un nom et un chiffre d'affaires à chaque segment obtenu
- transformer une segmentation en plan d'action budgété

## Plus de cible

Depuis la séance 4.1, chaque problème avait un `y` : un montant à prévoir, un
départ à anticiper. On pouvait donc mesurer si on avait raison.

Aujourd'hui, on retourne chez le détaillant des blocs 2 et 3, et la question
change de nature :

> *« Combien de types de clients avons-nous, et à quoi ressemblent-ils ? »*

Personne n'a étiqueté ces clients. C'est de l'apprentissage **non supervisé** —
et il n'y a **pas de bonne réponse**. Il y a des découpages plus ou moins
utiles, ce qui n'est pas la même chose.

### Les données : recency, frequency, monetary

Trois variables, un classique du marketing depuis quarante ans.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc4_ml/data/"

In [ ]:
cli = pd.read_csv(BASE + "clients_rfm.csv")   ## une ligne = un client

print(cli.shape)
cli.head(3)

In [ ]:
cli[["recence", "freq", "montant"]].describe().round(1)   ## les echelles

`recence` = jours depuis le dernier achat · `freq` = nombre de commandes ·
`montant` = total dépensé.

Notez les échelles : la récence va de 0 à 372, le montant de 30 à 143 825. **Un
facteur mille entre les deux.** Retenez-le, c'est le sujet du paragraphe
suivant.

## 1. Pourquoi standardiser — la démonstration

`KMeans` regroupe ce qui est **proche**. Voyons ce que ça donne sur les
colonnes brutes.

In [ ]:
# Sur les colonnes BRUTES, sans mise a l'echelle : volontairement faux
brut = KMeans(n_clusters=4, n_init=10, random_state=42).fit(cli[["recence", "freq", "montant"]])

cli.assign(g=brut.labels_).groupby("g").agg(
    n=("client_id", "size"), recence=("recence", "median"),
    freq=("freq", "median"), montant=("montant", "median")).round(1)

**Un groupe de 401 clients, et un groupe de deux.** Ce découpage est
inutilisable.

Pourquoi ? Un écart de 100 000 € sur le montant écrase complètement un écart
de 300 jours sur la récence. L'algorithme n'a regardé qu'une seule variable.

### Deux corrections, dans cet ordre

In [ ]:
variables = ["recence", "freq", "montant"]

# 1. log1p ecrase les valeurs extremes (log1p, et non log : il accepte le 0)
# 2. StandardScaler ramene chaque variable a la meme echelle
Xs = StandardScaler().fit_transform(np.log1p(cli[variables]))

print("moyennes apres mise a l'echelle :", Xs.mean(axis=0).round(2))   ## 0
print("ecarts-types                    :", Xs.std(axis=0).round(2))    ## 1

Chaque variable a maintenant une moyenne de 0 et un écart-type de 1 : elles
pèsent le même poids dans le calcul des distances.

> ⚠️ **Sans cette étape, une segmentation se forme sur la variable qui a les
> plus gros nombres.** C'est l'erreur la plus fréquente du clustering, et elle
> ne produit aucun message d'erreur.

## 2. Combien de groupes ?

In [ ]:
resultats = []
for k in range(2, 9):   ## de 2 a 8 groupes
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs)
    resultats.append({"k": k,
                      "inertie": round(km.inertia_, 1),   ## compacite des groupes
                      "silhouette": round(silhouette_score(Xs, km.labels_), 3)})

pd.DataFrame(resultats).set_index("k")

In [ ]:
pd.DataFrame(resultats).set_index("k")["inertie"].plot(marker="o", figsize=(7, 4))
plt.title("La courbe du coude")
plt.ylabel("inertie")
plt.show()

L'**inertie** mesure la compacité des groupes. Elle baisse toujours quand `k`
augmente — à `k` = 472, chaque client est son propre groupe et elle vaut zéro.
On cherche donc le **coude** : l'endroit où elle cesse de baisser vite.

La **silhouette** mesure autre chose : chaque point est-il plus proche de son
groupe que du groupe voisin ? Elle vaut ici **0,418 pour k = 2**, et décroît
ensuite.

### Et pourtant, nous retiendrons 4

Deux segments, ce sont « les bons » et « les autres » : aucune action ne s'en
déduit. Quatre segments donnent quatre traitements différents, et c'est ce
qu'on demande à une segmentation.

> 💡 **Le nombre de groupes n'est pas une question purement mathématique.**
> Les indicateurs cadrent la décision, l'usage la tranche.

## 3. Quatre segments, quatre noms

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Xs)
cli["groupe"] = km.labels_   ## un numero de groupe par client

profils = cli.groupby("groupe").agg(
    n=("client_id", "size"),
    recence=("recence", "median"),    ## mediane : robuste, cf. seance 3.1
    freq=("freq", "median"),
    montant=("montant", "median"),
    ca=("montant", "sum"),            ## somme ici : on veut le CA du segment
)
profils["part_ca"] = (100 * profils["ca"] / cli["montant"].sum()).round(1)
profils.round(1)

Chaque ligne se nomme toute seule :

| Profil | Nom |
|---|---|
| 4 jours, 9 commandes, 4 310 € | **les champions** |
| 46 jours, 4 commandes, 1 592 € | **les fidèles** |
| 23 jours, 1 commande, 332 € | **les nouveaux** |
| 186 jours, 1 commande, 429 € | **les dormants** |

Et la colonne qui change tout : **69 champions font 61 % du chiffre
d'affaires**.

C'est le constat de concentration de la séance 2.3 — deux clients irlandais,
22,7 % du CA — retrouvé par un chemin entièrement différent. Trois blocs, trois
méthodes, une même réalité : cette entreprise repose sur une poignée de
comptes.

In [ ]:
for g in sorted(cli["groupe"].unique()):
    part = cli.query("groupe == @g")
    plt.scatter(part["recence"], part["freq"], alpha=0.6, label=f"groupe {g}")

plt.yscale("log")   ## sans elle, le client a 201 commandes ecrase la figure
plt.xlabel("jours depuis le dernier achat")
plt.ylabel("nombre de commandes (echelle log)")
plt.title("Les quatre segments")
plt.legend()
plt.show()

## 4. Ce qu'on en fait — le plan d'action

In [ ]:
# On identifie par le COMPORTEMENT : les numeros de groupe ne sont pas stables
dormants = cli.query("groupe == @profils['recence'].idxmax()")
nouveaux = cli.query("groupe == @profils['freq'].idxmin() and groupe != @profils['recence'].idxmax()")

print("dormants :", len(dormants), "clients,",
      round(dormants["montant"].sum(), 2), "euros deja depenses")
print("nouveaux :", len(nouveaux), "clients, une seule commande chacun ou presque")

**L'arbitrage :** vous avez 5 000 € de budget de relance.

- **Réveiller les 149 dormants.** Ils ont déjà dépensé 74 683 € : ils savent
  commander, ils connaissent le catalogue. Mais six mois de silence, c'est
  souvent un client déjà parti ailleurs.
- **Convertir les 98 nouveaux.** Une seule commande chacun. Les faire passer à
  la deuxième est le geste qui transforme un acheteur en client.

Il n'y a pas de réponse mathématique. Il y a une décision à prendre avec des
chiffres — et c'est exactement ce qu'on attend de vous.

> 📌 **Ce que le bloc 4 ne sait pas faire.** Rien ici ne dit qu'une relance
> *marche*. Pour l'établir, il faut relancer un groupe et pas l'autre, puis
> comparer. C'est l'objet du bloc 5.

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| écraser les valeurs extrêmes | `np.log1p(df[variables])` |
| mettre les variables à la même échelle | `StandardScaler().fit_transform(...)` |
| former k groupes | `KMeans(n_clusters=4, n_init=10, random_state=42).fit(X)` |
| l'étiquette de chaque individu | `km.labels_` |
| la compacité des groupes | `km.inertia_` |
| la qualité de la séparation | `silhouette_score(X, km.labels_)` |
| décrire les groupes | `df.groupby("groupe").agg(...)` |

## Supervisé ou non ?

| | Supervisé (4.1 à 4.3) | Non supervisé (4.4) |
|---|---|---|
| On dispose de... | une cible connue | rien d'autre que les variables |
| On mesure la qualité par... | l'erreur sur un jeu de test | la cohérence des groupes, et l'usage qu'on en fait |
| La bonne réponse... | existe | **n'existe pas** — il y a des découpages plus ou moins utiles |

## Les trois phrases à retenir

1. **Standardiser n'est pas optionnel.** Sans mise à l'échelle, les groupes se
   forment sur la variable qui a les plus gros nombres — ici le montant, et
   deux clients se retrouvent seuls dans leur segment.

2. **Le nombre de groupes n'est pas une question purement mathématique.** La
   silhouette préfère 2 ; l'usage commercial en demande 4. Les deux se
   défendent, et c'est à vous d'arbitrer.

3. **Une segmentation sans nom ni budget ne sert à rien.** « Groupe 0 » n'est
   pas un livrable ; « 69 champions qui font 61 % du chiffre d'affaires » en
   est un.